In [1]:
# Imports & Configuration
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interactive, FloatSlider, BoundedFloatText, FloatText, VBox, HBox, Label, Layout, Button
from scipy.linalg import expm
import plotly.io as pio

pio.renderers.default = 'jupyterlab'  # Use 'notebook' for classic Jupyter

In [2]:
# Mathematical Helpers
I2 = np.eye(2, dtype=complex)
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
PAULIS = np.array([I2, sx, sy, sz])

def b_vec(s):
    return np.array([0.0, -np.cos(np.pi*s/2), -np.sin(np.pi*s/2)])

def b_dot_vec(s):
    k = np.pi/2
    return np.array([0.0, k*np.sin(np.pi*s/2), -k*np.cos(np.pi*s/2)])

def P0_from_b(b): return 0.5*(I2 - b[0]*sx - b[1]*sy - b[2]*sz)
def P1_from_b(b): return 0.5*(I2 + b[0]*sx + b[1]*sy + b[2]*sz)
def P0_dot(bd): return -0.5*(bd[0]*sx + bd[1]*sy + bd[2]*sz)

def build_L1_generator(s, gamma):
    b  = b_vec(s)
    bd = b_dot_vec(s)
    lam01_inv = 1.0 / (-gamma + 1j)
    
    P0, P1, P0d = P0_from_b(b), P1_from_b(b), P0_dot(bd)
    A = P0d @ P1 * lam01_inv
    H1 = 1j * (A.conj().T - A)
    
    alpha = (gamma / (1 + gamma**2)) * np.dot(bd, bd)
    f = alpha * s
    
    G = np.zeros((4, 4))
    for j in range(4):
        rho_j = PAULIS[j] / 2.0
        comm = H1 @ rho_j - rho_j @ H1
        diss = P1 @ rho_j @ P1 - 0.5*(P0 @ rho_j + rho_j @ P0)
        L1_rho = -1j * comm + f * diss
        for i in range(4):
            G[i, j] = np.real(np.trace(PAULIS[i] @ L1_rho))
    return G

def ode_rhs(s, n, eps, gamma):
    b = b_vec(s)
    return (np.cross(b, n) + gamma*np.cross(b, np.cross(b, n))) / eps

def solve_ode(eps, gamma, n0, s_max, N=2000):
    s_vals = np.linspace(0, s_max, N+1)
    n = np.zeros((N+1, 3)); n[0] = n0; h = s_max/N
    for i in range(N):
        s = s_vals[i]
        k1 = ode_rhs(s, n[i], eps, gamma)
        k2 = ode_rhs(s+h/2, n[i]+h/2*k1, eps, gamma)
        k3 = ode_rhs(s+h/2, n[i]+h/2*k2, eps, gamma)
        k4 = ode_rhs(s+h, n[i]+h*k3, eps, gamma)
        n[i+1] = n[i] + h/6*(k1+2*k2+2*k3+k4)
    return s_vals, n

def solve_stationary(s_max, N=1000):
    s = np.linspace(0, s_max, N+1)
    return s, -np.array([b_vec(si) for si in s])

def solve_slow(eps, gamma, s_max, N=1000):
    s = np.linspace(0, s_max, N+1)
    coef = eps/(1+gamma**2); alpha = (gamma/(1+gamma**2))*(np.pi/2)**2
    return s, np.array([
        -b_vec(si) + coef*(gamma*b_dot_vec(si) + np.cross(b_vec(si), b_dot_vec(si)) + b_vec(si)*alpha*si)
        for si in s
    ])

def solve_state_preserving(eps, gamma, s_max, N=1000):
    s = np.linspace(0, s_max, N+1)
    traj = np.zeros((N+1, 3))
    for idx, si in enumerate(s):
        G = build_L1_generator(si, gamma)
        ExpG = expm(eps * G)
        b = b_vec(si)
        v = ExpG @ np.array([1.0, -b[0], -b[1], -b[2]])
        traj[idx] = v[1:]
    return s, traj

# Helper functions for buttons
def get_sp_initial_value(gamma, eps):
    """Compute the initial Bloch vector of the state-preserving slow manifold at s=0."""
    s = 0.0
    G = build_L1_generator(s, gamma)
    ExpG = expm(eps * G)
    b = b_vec(s)  # [0, -1, 0] at s=0
    v0 = np.array([1.0, -b[0], -b[1], -b[2]])  # [1, 0, 1, 0]
    v = ExpG @ v0
    return v[1:]  # Return Bloch vector (nx, ny, nz)

def bloch_to_spherical(n):
    """Convert Bloch vector to spherical coordinates (r, theta, phi)."""
    r = np.linalg.norm(n)
    if r < 1e-10:
        return 0.0, 0.0, 0.0
    theta = np.arccos(np.clip(n[2] / r, -1.0, 1.0))
    phi = np.arctan2(n[1], n[0])
    if phi < 0:
        phi += 2*np.pi
    return r, theta, phi

def set_initial_to_stationary(b=None):
    """Button callback: set orbit initial condition to stationary axis at s=0."""
    # Stationary initial: -b_vec(0) = [0, 1, 0]
    widgets.children[2].value = 1.0          # r0
    widgets.children[3].value = np.pi/2      # theta0
    widgets.children[4].value = np.pi/2      # phi0

def set_initial_to_sp_slow(b=None):
    """Button callback: set orbit initial condition to state-preserving slow manifold at s=0."""
    eps_val = widgets.children[0].value
    gamma_val = widgets.children[1].value
    n_sp0 = get_sp_initial_value(gamma_val, eps_val)
    r0, theta0, phi0 = bloch_to_spherical(n_sp0)
    widgets.children[2].value = r0
    widgets.children[3].value = theta0
    widgets.children[4].value = phi0

In [3]:
# FigureWidget Setup
u = np.linspace(0, 2*np.pi, 40); v = np.linspace(0, np.pi, 20)
xs = np.outer(np.cos(u), np.sin(v)); ys = np.outer(np.sin(u), np.sin(v)); zs = np.outer(np.ones(len(u)), np.cos(v))

fig = go.FigureWidget()
fig.add_trace(go.Scatter3d(mode='lines', name='Orbit', line=dict(width=2, color='blue'), x=[], y=[], z=[]))
fig.add_trace(go.Mesh3d(x=xs.flatten(), y=ys.flatten(), z=zs.flatten(), alphahull=0, opacity=0.15, color='lightgray', name='Bloch Ball'))
fig.add_trace(go.Scatter3d(mode='lines', name='Stationary States', line=dict(width=2, color='red'), x=[], y=[], z=[]))
fig.add_trace(go.Scatter3d(mode='lines', name='Slow Manifold', line=dict(width=2, color='green', dash='dot'), x=[], y=[], z=[]))
fig.add_trace(go.Scatter3d(mode='lines', name='State-Preserving Slow', line=dict(width=2, color='magenta'), x=[], y=[], z=[]))

fig.update_layout(
    # title='Adiabatic Dynamics: γ=0.00, ε=0.050, s∈[0,1]',
    width=1100, height=850,
    scene=dict(
        xaxis=dict(range=[-1.2, 1.2], title='X'), yaxis=dict(range=[-1.2, 1.2], title='Y'),
        zaxis=dict(range=[-1.2, 1.2], title='Z'), aspectratio=dict(x=1, y=1, z=1),
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
    ),
    margin=dict(l=0, r=0, b=0, t=50)
)

def update_plot(eps=0.05, gamma=0.0, r0=1.0, theta0=np.pi/2, phi0=np.pi/2, s_max=1.0):
    n0 = r0*np.array([np.sin(theta0)*np.cos(phi0), np.sin(theta0)*np.sin(phi0), np.cos(theta0)])
    _, n_ode = solve_ode(eps, gamma, n0, s_max, N=2000)
    _, n_stat = solve_stationary(s_max, N=1000)
    _, n_slow = solve_slow(eps, gamma, s_max, N=1000)
    _, n_sp   = solve_state_preserving(eps, gamma, s_max, N=1000)
    
    with fig.batch_update():
        fig.data[0].x, fig.data[0].y, fig.data[0].z = n_ode[:,0], n_ode[:,1], n_ode[:,2]
        fig.data[2].x, fig.data[2].y, fig.data[2].z = n_stat[:,0], n_stat[:,1], n_stat[:,2]
        fig.data[3].x, fig.data[3].y, fig.data[3].z = n_slow[:,0], n_slow[:,1], n_slow[:,2]
        fig.data[4].x, fig.data[4].y, fig.data[4].z = n_sp[:,0], n_sp[:,1], n_sp[:,2]
        #fig.layout.title = f'γ={gamma:.2f}, ε={eps:.3f}, s∈[0,{s_max:.1f}]'

In [4]:
# Interactive Widgets
widgets = interactive(
    update_plot,
    eps=FloatSlider(min=0.0001, max=0.1, step=0.0001, value=0.05, description='ε', continuous_update=True, layout=Layout(width='380px')),
    gamma=FloatSlider(min=0.0, max=1.0, step=0.01, value=0.0, description='γ', continuous_update=True, layout=Layout(width='380px')),
    r0=FloatSlider(min=0.0, max=1.0, step=0.01, value=1.0, description='r₀', continuous_update=True, layout=Layout(width='380px')),
    theta0=FloatSlider(min=0.0, max=np.pi, step=0.01, value=np.pi/2, description='θ₀', continuous_update=True, layout=Layout(width='380px')),
    phi0=FloatSlider(min=0.0, max=2*np.pi, step=0.01, value=np.pi/2, description='ϕ₀', continuous_update=True, layout=Layout(width='380px')),
    s_max=BoundedFloatText(value=1.0, min=0.1, max=10.0, step=0.1, description='s_max:', layout=Layout(width='300px'))
)

# Max controls for sliders
eps_max_ctrl = BoundedFloatText(value=0.1, min=0.01, max=1.0, step=0.01, description='ε_max:', layout=Layout(width='200px'))
gamma_max_ctrl = BoundedFloatText(value=1.0, min=0.1, max=10.0, step=0.1, description='γ_max:', layout=Layout(width='200px'))

def on_eps_max(change): widgets.children[0].max = change['new']; widgets.children[0].value = min(widgets.children[0].value, change['new'])
def on_gamma_max(change): widgets.children[1].max = change['new']; widgets.children[1].value = min(widgets.children[1].value, change['new'])

eps_max_ctrl.observe(on_eps_max, names='value')
gamma_max_ctrl.observe(on_gamma_max, names='value')

btn_stationary = Button(
    description='Set to Stationary Initial', 
    button_style='info', 
    tooltip='Set orbit start to -b̂(0) = [0,1,0]',
    layout=Layout(width='220px')
)
btn_sp = Button(
    description='Set to SP Slow Initial', 
    button_style='success', 
    tooltip='Set orbit start to exp(εL₁(0))P₀(0)',
    layout=Layout(width='220px')
)

btn_stationary.on_click(set_initial_to_stationary)
btn_sp.on_click(set_initial_to_sp_slow)

btn_row = HBox([btn_stationary, btn_sp], layout=Layout(justify_content='center', padding='8px 0'))

# Controls Layout
controls = VBox([
    HBox([widgets.children[0], eps_max_ctrl]),
    HBox([widgets.children[1], gamma_max_ctrl]),
    widgets.children[2], widgets.children[3], widgets.children[4],
    HBox([Label('Max time:'), widgets.children[5]]),
    btn_row  # NEW: Button row inserted here
], layout=Layout(padding='12px', border='1px solid #e0e0e0', border_radius='8px'))

In [5]:
# Display & Initialization
display(VBox([controls, fig], layout=Layout(align_items='center', width='100%')))
update_plot()  # Trigger initial render